# Parkinson’s Disease Progression — Independent Reproduction Plan

**课程允许的用途：方法讨论、API 学习、代码调试。本文是独立实现骨架，不是完整可提交的模型。**

依据 [2023 Kaggle 冠军方法说明](https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/writeups/connecting-dotts-1st-place-solution) 进行概念级复现，**不要复制原 Kaggle Notebook、之前的 V7 文件或此骨架中的占位部分直接提交**。

复现目标：用时间/访问模式特征预测 UPDRS1–4（0/6/12/24 月），比较离散概率 LightGBM、直接优化 SMAPE+1 的轻量 MLP 和二者的简单平均。原冠军不使用蛋白 NPX 原始测量值；这是其公开方案的重要特点，不代表所有其他设置都无效。

组内独立完成记录：在最终报告填写每段核心代码的组员作者、设计选择、数据泄漏检查、CV 与 Kaggle 的具体版本号；如实披露 ChatGPT 用于 API/调试/说明的范围。

## 1. 文件与任务定义

数据表：`train_clinical_data.csv`、`supplemental_clinical_data.csv`、`train_proteins.csv`、`train_peptides.csv`。预测目标为 `updrs_1`–`updrs_4`，预测距离为 0、6、12、24 月。

请组员确认补充数据 `visit_month=5` 的处理方式，不能静默套用他人代码。下面单元仅提供**文件读取和断言的 API 示例**。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/kaggle/input/amp-pd")  # 本地调试时更换为自己的数据路径
TARGETS = tuple(f"updrs_{i}" for i in range(1, 5))
HORIZONS = (0, 6, 12, 24)
KEY = ["patient_id", "visit_month"]

# TODO (group member A): verify actual competition directory and file schema.
# clinical = pd.read_csv(DATA / "train_clinical_data.csv")
# supplemental = pd.read_csv(DATA / "supplemental_clinical_data.csv")
# proteins = pd.read_csv(DATA / "train_proteins.csv", usecols=KEY)
# peptides = pd.read_csv(DATA / "train_peptides.csv", usecols=KEY)


## 2. 标签构造与输入可见性

以 `(patient_id, origin_visit_month, target, horizon)` 为请求键；评分对应 `target_month = origin_visit_month + horizon`。避免把实际目标评分、未到达的未来访问记录或隐藏患者的完整访问历史作为特征。所选患者的训练与验证数据必须彻底分开。

冠军列出的特征概念：起点月、预测距离、目标月、是否采血、是否补充临床数据、6/18/48 月的已观测访问标记、此前 6/18 月非年度访问次数、UPDRS 目标索引。

组员必须自己定义在线 API 逐批到达时，哪些访问标记能够被观测。**离线 CV 必须模拟同样的观测条件**。

In [ ]:
def build_requests_and_labels(clinical, supplemental, observed_origins):
    """Student implementation required; output request and label frames.

    Contract: request keys (patient_id, visit_month, target, horizon),
    target_month and numeric rating; no label information in feature columns.
    """
    raise NotImplementedError("Implement patient/time/target joins independently")


def build_time_features(requests, observed_visits, observed_blood_visits):
    """Student implementation required; use ONLY visits observed by origin month."""
    raise NotImplementedError("Implement causal feature construction independently")


## 3. SMAPE+1 评分与测试

比赛指标每条样本的误差为 `200 * abs(y - pred) / (abs(y) + abs(pred) + 2)`；UPDRS 目标和预测需为非负值。**下面是指标用法示例，不是模型训练实现。**

In [ ]:
def smape_plus_one(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    if y.shape != pred.shape or not np.isfinite(pred).all():
        raise ValueError("Invalid prediction shape or non-finite values")
    return float(np.mean(200.0 * np.abs(y - pred) / (np.abs(y) + np.abs(pred) + 2.0)))

assert smape_plus_one([0, 5], [0, 5]) == 0
assert smape_plus_one([0], [1]) > 0


## 4. LightGBM 概率预测与 SMAPE 决策（由组员独立完成）

冠军将 UPDRS 值视为整数类别，预测 0–86 的类别概率，再选择**最小期望 SMAPE+1**的整数输出。实现时验证类别缺失、概率行归一化、类别顺序，以及训练集评分上限。

请组员 B 完成分类器训练和预测后，将输出转换为 `n_samples × n_classes` 概率矩阵；先单独测试解码器，再纳入 CV。

In [ ]:
def fit_lgbm_distribution(x_train, y_train, *, random_seed=4486):
    """TODO: implement multiclass LightGBM based on independent choices."""
    raise NotImplementedError("Team implements and explains its own training code")


def choose_smape_minimizer(class_probabilities, class_labels):
    """TODO: select integer score minimizing expected SMAPE+1."""
    raise NotImplementedError("Team implements and tests the decision rule")


## 5. 小型神经网络基线（由组员独立完成）

冠军描述为前馈神经网络，使用 SMAPE+1 作为损失；末端激活需避免负预测。需要自己决定网络宽度、优化器、训练 epochs、特征缩放和随机种子，并记录这些选择的依据。不得直接从公开 Notebook 复制超参数作为“独立设计”。

In [ ]:
def train_regression_mlp(x_train, y_train, x_valid, *, random_seed=4486):
    """TODO: implement own MLP, SMAPE+1 loss, proper train-only scaling."""
    raise NotImplementedError("Team implements and documents the training loop")


## 6. 患者级验证、模型集成与残差校正

使用患者为分组键的分折验证；同一患者不得出现在训练与验证两侧。先核算 LightGBM、MLP、二者平均的 SMAPE+1。为新增残差校正器生成**真正患者级 OOF 基线预测**，不可在基线训练数据上计算残差后自称独立预测。

新模型目标可以是：`baseline_prediction + bounded_residual(features)`，需单独证明改进和失败情况；这是与原冠军复现区分的实验增量。

In [ ]:
def evaluate_patient_folds(data, patient_fold_map):
    """TODO: train only on other patients; report per-fold and OOF metrics.

    Must compare: LightGBM, MLP, average, optional independent improvement.
    """
    raise NotImplementedError("Team owns its CV implementation")


## 7. Kaggle 官方批次接口检查

最终提交必须编辑**已有官方 demo Notebook**，保留原 `amp_pd_peptide.make_env()` → `iter_test()` → `env.predict()` 工作流、`docker_image_pinning_type=original`、Internet disabled。请先在官方 demo 中确认每个 batch 的四个对象列名和数据可见性，并测试返回 DataFrame 与 sample 的顺序完全一致。

这里不包含自动提交代码：CS4486 禁止 AI 生成完整 Kaggle Notebook 并原样提交。

In [ ]:
def predict_one_official_batch(test, test_peptides, test_proteins, sample_submission):
    """TODO: parse prediction_id; update observable state; predict in original order."""
    raise NotImplementedError("Team completes this after auditing official demo")


## 8. 复现证据检查清单

- [ ] 每项核心实现由组员编写、理解并审查；记录作者和 AI 辅助范围。
- [ ] 训练集、验证集患者 ID 无交集。
- [ ] 访问历史和蛋白存在性特征不读取未来或隐藏信息。
- [ ] 类别概率合法且决策输出非负。
- [ ] 记录每折 SMAPE+1、随机种子、LightGBM/MLP 参数和运行版本。
- [ ] 明确区分冠军方案思路、组内独立复现、你们的新增残差校正实验。
- [ ] 只有实际 Kaggle 提交返回分数后，才记录 Public/Private 成绩。

参考：2023 冠军 [1st Place Solution](https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/writeups/connecting-dotts-1st-place-solution)；你们课程的 `Mini Project Instruction-CS4486.pdf` 第 V 节学术诚信要求。